# 03_build_parent_child_chunks

## 목적

이 노트북은 `02_parse_regulation_articles.ipynb`에서 생성한 `parents.jsonl`을 읽고,
조문 parent를 검색용 child chunk로 분리한다.

입력:
- data/retrieval/parents.jsonl

출력:
- data/retrieval/children.jsonl
- data/retrieval/debug_child_chunks/child_chunk_summary.json

이번 노트북에서 하는 일:

1. parents.jsonl 로드
2. 조문 본문에서 항/호/목 패턴 탐지
3. parent article을 child chunk로 분리
4. child chunk에 parent_id, law_name, article_no, page metadata 유지
5. risk_tags, keywords를 간단한 rule 기반으로 부여
6. children.jsonl 저장 및 재로드 검증

이번 노트북에서는 아직 ChromaDB를 만들지 않는다.
이번 노트북에서는 아직 BM25 index를 만들지 않는다.

In [1]:
# 기본 라이브러리 및 경로 설정
from pathlib import Path
import re
import json
import hashlib
from pprint import pprint
from typing import List, Dict, Any, Tuple
from collections import Counter, defaultdict

import pandas as pd


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"
PARENTS_PATH = RETRIEVAL_DIR / "parents.jsonl"
CHILDREN_PATH = RETRIEVAL_DIR / "children.jsonl"
DEBUG_DIR = RETRIEVAL_DIR / "debug_child_chunks"

DEBUG_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_NAME = "complypilot_regulations_v2"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PARENTS_PATH:", PARENTS_PATH)
print("CHILDREN_PATH:", CHILDREN_PATH)
print("DEBUG_DIR:", DEBUG_DIR)
print("COLLECTION_NAME:", COLLECTION_NAME)

PROJECT_ROOT: c:\Users\USER\Desktop\complypilot-jb
PARENTS_PATH: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\parents.jsonl
CHILDREN_PATH: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\children.jsonl
DEBUG_DIR: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\debug_child_chunks
COLLECTION_NAME: complypilot_regulations_v2


In [2]:
# JSONL 로드/저장 함수
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """
    JSONL 파일을 읽어 dict 리스트로 반환합니다.

    Args:
        path: JSONL 파일 경로

    Return:
        dict 리스트
    """
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    return rows


def save_jsonl(rows: List[Dict[str, Any]], path: Path) -> None:
    """
    dict 리스트를 JSONL 파일로 저장합니다.

    Args:
        rows: 저장할 dict 리스트
        path: 저장 경로

    Return:
        None
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


parents = load_jsonl(PARENTS_PATH)

print("parent row 수:", len(parents))
pprint(parents[0] if parents else None)

parent row 수: 608
{'article_no': '제1조',
 'article_title': '목적',
 'doc_code': 'financial_consumer_supervisory_regulation',
 'document_priority': 3,
 'document_type': 'supervisory_regulation',
 'effective_date': '2026.4.2.',
 'has_article_header': True,
 'is_short_article': False,
 'law_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)',
 'page_end': 1,
 'page_start': 1,
 'parent_id': 'financial_consumer_supervisory_regulation__article_1',
 'parse_status': 'ok',
 'source_file': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
 'text': '제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한\n'
         '사항을 규정함을 목적으로 한다.',
 'text_length': 81}


In [3]:
# parents 기본 품질 확인
df_parents = pd.DataFrame(parents)

print("parents shape:", df_parents.shape)

display_cols = [
    "law_name",
    "document_type",
    "article_no",
    "article_title",
    "page_start",
    "page_end",
    "text_length",
    "parent_id",
]

display(df_parents[display_cols].head(20))

print("document_type 분포")
display(df_parents["document_type"].value_counts().reset_index())

print("parse_status 분포")
if "parse_status" in df_parents.columns:
    display(df_parents["parse_status"].value_counts().reset_index())

parents shape: (608, 16)


,law_name,document_type,article_no,article_title,page_start,page_end,text_length,parent_id
0,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제1조,목적,1,1,81,financial_consumer_supervisory_regulation__art...
1,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,1,3,3430,financial_consumer_supervisory_regulation__art...
2,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제3조,금융상품의 유형,3,4,464,financial_consumer_supervisory_regulation__art...
3,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제4조,금융회사등의 업종구분,4,4,102,financial_consumer_supervisory_regulation__art...
4,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제5조,금융상품자문업자의 등록요건,4,5,1460,financial_consumer_supervisory_regulation__art...
5,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제6조,금융상품판매대리ㆍ중개업자의 등록요건,5,6,1744,financial_consumer_supervisory_regulation__art...
6,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제7조,등록신청,6,6,1034,financial_consumer_supervisory_regulation__art...
7,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제8조,등록수수료,6,7,153,financial_consumer_supervisory_regulation__art...
8,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제9조,내부통제기준,7,7,889,financial_consumer_supervisory_regulation__art...
9,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제10조,적합성 원칙,7,9,1861,financial_consumer_supervisory_regulation__art...


document_type 분포


,document_type,count
0,law,297
1,supervisory_regulation,258
2,enforcement_decree,53


parse_status 분포


,parse_status,count
0,ok,583
1,short_text_review,25


In [4]:
# Child split 패턴 정의
CIRCLED_PARAGRAPH_PATTERN = re.compile(
    r"(?=(?:^|\n)\s*[①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳])"
)

NUMBERED_ITEM_PATTERN = re.compile(
    r"(?=(?:^|\n)\s*\d+\.\s*)"
)

KOREAN_SUBITEM_PATTERN = re.compile(
    r"(?=(?:^|\n)\s*[가-하]\.\s*)"
)

ARTICLE_HEADER_AT_START_PATTERN = re.compile(
    r"^\s*제\s*\d+\s*조(?:의\s*\d+)?\s*\([^)]*\)"
)


def normalize_chunk_text(text: str) -> str:
    """
    child chunk 저장 전 텍스트를 정리합니다.

    Args:
        text: 원본 텍스트

    Return:
        정리된 텍스트
    """
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def remove_article_header(text: str) -> Tuple[str, str]:
    """
    조문 본문 앞의 제○조(제목) 헤더를 분리합니다.

    Args:
        text: parent article text

    Return:
        article_header, body
    """
    match = ARTICLE_HEADER_AT_START_PATTERN.search(text)

    if not match:
        return "", text.strip()

    header = match.group(0).strip()
    body = text[match.end():].strip()

    return header, body

In [5]:
# 텍스트 split 함수
def split_by_pattern(text: str, pattern: re.Pattern) -> List[str]:
    """
    정규식 lookahead 패턴을 기준으로 텍스트를 나눕니다.

    Args:
        text: 입력 텍스트
        pattern: split 기준 정규식

    Return:
        분리된 텍스트 리스트
    """
    parts = pattern.split(text)
    parts = [normalize_chunk_text(part) for part in parts if normalize_chunk_text(part)]
    return parts


def choose_split_strategy(body: str) -> Tuple[str, List[str]]:
    """
    parent article body를 child chunk로 나눌 전략을 선택합니다.

    Args:
        body: 조문 헤더를 제거한 본문

    Return:
        split_strategy, child_texts
    """
    body = normalize_chunk_text(body)

    circled_parts = split_by_pattern(body, CIRCLED_PARAGRAPH_PATTERN)
    circled_count = sum(
        1 for part in circled_parts
        if re.match(r"^\s*[①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳]", part)
    )

    if circled_count >= 2:
        return "paragraph_circled", circled_parts

    numbered_parts = split_by_pattern(body, NUMBERED_ITEM_PATTERN)
    numbered_count = sum(
        1 for part in numbered_parts
        if re.match(r"^\s*\d+\.\s*", part)
    )

    if numbered_count >= 2:
        return "numbered_item", numbered_parts

    korean_parts = split_by_pattern(body, KOREAN_SUBITEM_PATTERN)
    korean_count = sum(
        1 for part in korean_parts
        if re.match(r"^\s*[가-하]\.\s*", part)
    )

    if korean_count >= 2:
        return "korean_subitem", korean_parts

    return "whole_article", [body]

In [6]:
# marker 추출 함수
def extract_unit_markers(child_text: str) -> Dict[str, str]:
    """
    child chunk 시작 부분에서 항/호/목 marker를 추출합니다.

    Args:
        child_text: child chunk 본문

    Return:
        paragraph_no, item_no, subitem_no
    """
    text = child_text.strip()

    paragraph_no = ""
    item_no = ""
    subitem_no = ""

    paragraph_match = re.match(
        r"^([①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳])",
        text,
    )
    if paragraph_match:
        paragraph_no = paragraph_match.group(1)

    item_match = re.match(r"^(\d+)\.\s*", text)
    if item_match:
        item_no = f"제{item_match.group(1)}호"

    subitem_match = re.match(r"^([가-하])\.\s*", text)
    if subitem_match:
        subitem_no = f"{subitem_match.group(1)}목"

    return {
        "paragraph_no": paragraph_no,
        "item_no": item_no,
        "subitem_no": subitem_no,
    }


test_texts = [
    "① 금융상품판매업자는 ...",
    "1. 대출성 상품에 관한 사항",
    "가. 이자율 및 수수료",
    "본문 전체",
]

for text in test_texts:
    print(text, "=>", extract_unit_markers(text))

① 금융상품판매업자는 ... => {'paragraph_no': '①', 'item_no': '', 'subitem_no': ''}
1. 대출성 상품에 관한 사항 => {'paragraph_no': '', 'item_no': '제1호', 'subitem_no': ''}
가. 이자율 및 수수료 => {'paragraph_no': '', 'item_no': '', 'subitem_no': '가목'}
본문 전체 => {'paragraph_no': '', 'item_no': '', 'subitem_no': ''}


In [7]:
# risk_tags / keywords 간단 부여
RISK_TAG_RULES = {
    "approval_misleading": ["승인", "보장", "확정", "누구나", "무조건"],
    "rate_condition_missing": ["금리", "이자율", "최저금리", "우대금리", "연"],
    "fee_missing": ["수수료", "비용", "중도상환", "연체", "부대비용"],
    "principal_guarantee_misleading": ["원금", "손실", "보장"],
    "return_misleading": ["수익", "수익률", "확정수익", "고수익"],
    "explanation_duty": ["설명", "설명의무", "중요사항", "고지"],
    "advertising_regulation": ["광고", "표시", "오인", "과장", "비교"],
    "unfair_solicitation": ["부당권유", "권유", "적합성", "적정성"],
}

PRODUCT_TYPE_RULES = {
    "loan": ["대출", "여신", "금리", "이자율", "상환"],
    "deposit": ["예금", "적금", "예금자보호"],
    "card": ["카드", "신용카드", "여신전문"],
    "investment": ["투자", "수익률", "금융투자", "원금손실"],
    "insurance": ["보험", "보험료", "보험금"],
}


def assign_risk_tags(text: str) -> List[str]:
    """
    rule 기반으로 risk_tags를 부여합니다.

    Args:
        text: child chunk 본문

    Return:
        risk tag 리스트
    """
    tags = []

    for tag, keywords in RISK_TAG_RULES.items():
        if any(keyword in text for keyword in keywords):
            tags.append(tag)

    return sorted(set(tags))


def assign_product_types(text: str) -> List[str]:
    """
    rule 기반으로 product_types를 부여합니다.

    Args:
        text: child chunk 본문

    Return:
        product type 리스트
    """
    product_types = []

    for product_type, keywords in PRODUCT_TYPE_RULES.items():
        if any(keyword in text for keyword in keywords):
            product_types.append(product_type)

    return sorted(set(product_types))


def extract_keywords(text: str, max_keywords: int = 12) -> List[str]:
    """
    검색 보조용 키워드를 간단히 추출합니다.

    Args:
        text: child chunk 본문
        max_keywords: 최대 키워드 수

    Return:
        키워드 리스트
    """
    candidate_keywords = []

    for keywords in list(RISK_TAG_RULES.values()) + list(PRODUCT_TYPE_RULES.values()):
        for keyword in keywords:
            if keyword in text:
                candidate_keywords.append(keyword)

    return sorted(set(candidate_keywords))[:max_keywords]

In [8]:
# child_id 생성 함수
def make_child_id(parent_id: str, child_index: int, child_text: str) -> str:
    """
    parent_id, child_index, child text 기반으로 child_id를 생성합니다.

    Args:
        parent_id: parent article id
        child_index: child 순번
        child_text: child 본문

    Return:
        child_id
    """
    digest = hashlib.md5(child_text.encode("utf-8")).hexdigest()[:8]
    return f"{parent_id}__child_{child_index:03d}_{digest}"

In [9]:
# parent 1개를 child chunks로 변환
def build_child_chunks_for_parent(parent: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    parent article 1개를 child chunks로 변환합니다.

    Args:
        parent: parent article row

    Return:
        child chunk row 리스트
    """
    parent_text = parent.get("text", "")
    article_header, body = remove_article_header(parent_text)

    split_strategy, child_texts = choose_split_strategy(body)

    child_rows = []

    for child_index, child_text in enumerate(child_texts, start=1):
        child_text = normalize_chunk_text(child_text)

        if not child_text:
            continue

        markers = extract_unit_markers(child_text)

        context_text = f"{article_header}\n{child_text}".strip()
        risk_tags = assign_risk_tags(context_text)
        product_types = assign_product_types(context_text)
        keywords = extract_keywords(context_text)

        child_id = make_child_id(parent["parent_id"], child_index, child_text)

        child_rows.append({
            "chunk_id": child_id,
            "child_id": child_id,
            "parent_id": parent["parent_id"],
            "child_index": child_index,
            "split_strategy": split_strategy,

            "doc_code": parent.get("doc_code", ""),
            "law_name": parent.get("law_name", ""),
            "document_type": parent.get("document_type", ""),
            "document_priority": parent.get("document_priority", 9),

            "article_no": parent.get("article_no", ""),
            "article_title": parent.get("article_title", ""),
            "paragraph_no": markers["paragraph_no"],
            "item_no": markers["item_no"],
            "subitem_no": markers["subitem_no"],

            "page_start": parent.get("page_start", 0),
            "page_end": parent.get("page_end", 0),
            "page": parent.get("page_start", 0),
            "effective_date": parent.get("effective_date", ""),
            "source_file": parent.get("source_file", ""),

            "risk_tags": risk_tags,
            "product_types": product_types,
            "keywords": keywords,

            "text": context_text,
            "child_text": child_text,
            "parent_text": parent_text,
            "text_length": len(context_text),
            "parent_text_length": len(parent_text),

            "parse_status": parent.get("parse_status", "ok"),
        })

    return child_rows


sample_parent = parents[0]
sample_children = build_child_chunks_for_parent(sample_parent)

print("sample parent:")
print(sample_parent["law_name"], sample_parent["article_no"], sample_parent["article_title"])
print("child 개수:", len(sample_children))
pprint(sample_children[:3])

sample parent:
금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제1조 목적
child 개수: 1
[{'article_no': '제1조',
  'article_title': '목적',
  'child_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
  'child_index': 1,
  'child_text': '이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한\n'
                '사항을 규정함을 목적으로 한다.',
  'chunk_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
  'doc_code': 'financial_consumer_supervisory_regulation',
  'document_priority': 3,
  'document_type': 'supervisory_regulation',
  'effective_date': '2026.4.2.',
  'item_no': '',
  'keywords': [],
  'law_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)',
  'page': 1,
  'page_end': 1,
  'page_start': 1,
  'paragraph_no': '',
  'parent_id': 'financial_consumer_supervisory_regulation__article_1',
  'parent_text': '제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 '
                 '필요한\n'
                 '사항을 규정함을 목적으로 한다.',
  'parent

In [10]:
# 샘플 child 본문 확인
for idx, child in enumerate(sample_children[:5], start=1):
    print("=" * 120)
    print(f"[{idx}] {child['law_name']} {child['article_no']}({child['article_title']})")
    print("chunk_id:", child["chunk_id"])
    print("parent_id:", child["parent_id"])
    print("strategy:", child["split_strategy"])
    print("paragraph_no:", child["paragraph_no"])
    print("item_no:", child["item_no"])
    print("subitem_no:", child["subitem_no"])
    print("risk_tags:", child["risk_tags"])
    print("keywords:", child["keywords"])
    print("-" * 120)
    print(child["text"][:1500])

[1] 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제1조(목적)
chunk_id: financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8
parent_id: financial_consumer_supervisory_regulation__article_1
strategy: whole_article
paragraph_no: 
item_no: 
subitem_no: 
risk_tags: []
keywords: []
------------------------------------------------------------------------------------------------------------------------
제1조(목적)
이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한
사항을 규정함을 목적으로 한다.


In [11]:
# 전체 parent를 children으로 변환
all_children = []
build_errors = []

for parent in parents:
    try:
        child_rows = build_child_chunks_for_parent(parent)
        all_children.extend(child_rows)
    except Exception as e:
        build_errors.append({
            "parent_id": parent.get("parent_id"),
            "law_name": parent.get("law_name"),
            "article_no": parent.get("article_no"),
            "error": str(e),
        })

print("parent 수:", len(parents))
print("child 수:", len(all_children))
print("build error 수:", len(build_errors))

if build_errors:
    pprint(build_errors[:10])

parent 수: 608
child 수: 2155
build error 수: 0


In [12]:
# children DataFrame 요약
df_children = pd.DataFrame(all_children)

print("children shape:", df_children.shape)

display_cols = [
    "law_name",
    "document_type",
    "article_no",
    "article_title",
    "child_index",
    "split_strategy",
    "paragraph_no",
    "item_no",
    "subitem_no",
    "text_length",
    "risk_tags",
    "keywords",
    "chunk_id",
    "parent_id",
]

display(df_children[display_cols].head(30))

print("split_strategy 분포")
display(df_children["split_strategy"].value_counts().reset_index())

print("document_type 분포")
display(df_children["document_type"].value_counts().reset_index())

children shape: (2155, 28)


,law_name,document_type,article_no,article_title,child_index,split_strategy,paragraph_no,item_no,subitem_no,text_length,risk_tags,keywords,chunk_id,parent_id
0,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제1조,목적,1,whole_article,,,,81,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
1,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,1,paragraph_circled,①,,,1033,[rate_condition_missing],"[금융투자, 대출, 보험, 상환, 여신, 여신전문, 연, 예금, 예금자보호, 투자]",financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
2,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,2,paragraph_circled,②,,,1075,[],"[금융투자, 대출, 보험, 상환, 카드, 투자]",financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
3,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,3,paragraph_circled,③,,,103,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
4,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,4,paragraph_circled,④,,,130,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
5,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,5,paragraph_circled,⑤,,,57,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
6,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,6,paragraph_circled,⑥,,,621,[],"[금융투자, 투자]",financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
7,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,7,paragraph_circled,⑦,,,91,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
8,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,8,paragraph_circled,⑧,,,290,[],"[금융투자, 투자]",financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...
9,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),supervisory_regulation,제2조,정의,9,paragraph_circled,⑨,,,86,[],[],financial_consumer_supervisory_regulation__art...,financial_consumer_supervisory_regulation__art...


split_strategy 분포


,split_strategy,count
0,paragraph_circled,1669
1,numbered_item,350
2,whole_article,136


document_type 분포


,document_type,count
0,law,1017
1,supervisory_regulation,908
2,enforcement_decree,230


In [13]:
# parent-child 참조 무결성 확인
parent_ids = set(row["parent_id"] for row in parents)
child_parent_ids = set(row["parent_id"] for row in all_children)

missing_parent_ids = sorted(child_parent_ids - parent_ids)

print("parent_id 수:", len(parent_ids))
print("child가 참조하는 parent_id 수:", len(child_parent_ids))
print("없는 parent_id 참조 수:", len(missing_parent_ids))

if missing_parent_ids:
    pprint(missing_parent_ids[:20])

assert len(missing_parent_ids) == 0, "children 중 parents에 없는 parent_id를 참조하는 row가 있습니다."

print("[OK] parent-child 참조 무결성 확인 완료")

parent_id 수: 608
child가 참조하는 parent_id 수: 608
없는 parent_id 참조 수: 0
[OK] parent-child 참조 무결성 확인 완료


In [14]:
# chunk_id 중복 확인
chunk_id_counts = Counter(row["chunk_id"] for row in all_children)
duplicated_chunk_ids = [
    chunk_id for chunk_id, count in chunk_id_counts.items()
    if count > 1
]

print("중복 chunk_id 수:", len(duplicated_chunk_ids))

if duplicated_chunk_ids:
    pprint(duplicated_chunk_ids[:10])

assert len(duplicated_chunk_ids) == 0, "중복 chunk_id가 있습니다."

print("[OK] chunk_id 유니크 확인 완료")

중복 chunk_id 수: 0
[OK] chunk_id 유니크 확인 완료


In [15]:
# 너무 짧거나 긴 child chunk 진단
def diagnose_child_lengths(children: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    child chunk 길이 분포를 진단합니다.

    Args:
        children: child chunk row 리스트

    Return:
        길이 진단 요약 dict
    """
    lengths = [row["text_length"] for row in children]

    if not lengths:
        return {}

    return {
        "child_count": len(lengths),
        "min_length": min(lengths),
        "max_length": max(lengths),
        "avg_length": round(sum(lengths) / len(lengths), 2),
        "short_lt_80": sum(1 for x in lengths if x < 80),
        "long_gt_3000": sum(1 for x in lengths if x > 3000),
        "long_gt_5000": sum(1 for x in lengths if x > 5000),
    }


length_summary = diagnose_child_lengths(all_children)
pprint(length_summary)

short_children = [row for row in all_children if row["text_length"] < 80]
long_children = [row for row in all_children if row["text_length"] > 3000]

print("짧은 child 샘플:", len(short_children))
for row in short_children[:5]:
    print(row["law_name"], row["article_no"], row["text_length"], row["text"][:200])

print("\n긴 child 샘플:", len(long_children))
for row in long_children[:5]:
    print(row["law_name"], row["article_no"], row["text_length"], row["text"][:200])

{'avg_length': 180.91,
 'child_count': 2155,
 'long_gt_3000': 1,
 'long_gt_5000': 0,
 'max_length': 3859,
 'min_length': 11,
 'short_lt_80': 472}
짧은 child 샘플: 472
금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제2조 57 제2조(정의)
⑤ 영 제2조제6항제6호에서 "금융위원회가 정하여 고시하는 자"란 신용협동조합을 말한다.
금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제3조 50 제3조(금융상품의 유형)
1. 영 제3조제1항제3호: 제2조제1항제1호에 해당하는 금융상품
금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제5조 74 제5조(금융상품자문업자의 등록요건)
③ 영 제5조제3항제1호에서 "금융위원회가 정하여 고시하는 비율"이란 100분의 200을 말한다.
금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제6조 75 제6조(금융상품판매대리ㆍ중개업자의 등록요건)
⑥ 영 제6조제2항제5호에서 "금융위원회가 정하여 고시하는 보증금"이란 5천만원을 말한다.
금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제8조 66 제8조(등록수수료)
영 제9조에서 "금융위원회가 정하여 고시하는 수수료"란 다음 각 호의 구분에 따른 금액을 말한
다.

긴 child 샘플: 1
은행업감독규정 (금융위원회고시)(제2026-10호)(20260401) 제26조 3859 제26조(경영지도비율)
① 은행은 법 제34조에 따라 다음 각 호에서 정하는 경영지도비율을 유지하여야 한다. 다만,
제3호의 경영지도비율은 직전분기말월의 원화대출금이 4조원 미만인 은행의 경우에는 적용하지 아니한다.<개정
2023. 7. 5.>
1. 자본비율에 관하여 다음 각 목에서 정하는 최소 준수비율


In [16]:
# 품질 플래그 추가
def add_child_quality_flags(children: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    child chunk에 품질 플래그를 추가합니다.

    Args:
        children: child chunk row 리스트

    Return:
        품질 플래그가 추가된 child chunk 리스트
    """
    updated = []

    for row in children:
        row = dict(row)
        text_length = row.get("text_length", 0)

        row["is_short_child"] = text_length < 80
        row["is_long_child"] = text_length > 3000
        row["has_parent_id"] = bool(row.get("parent_id"))
        row["has_article_no"] = bool(row.get("article_no"))
        row["has_page"] = bool(row.get("page_start") or row.get("page"))

        status = "ok"

        if row["is_short_child"]:
            status = "short_child_review"

        if row["is_long_child"]:
            status = "long_child_review"

        if not row["has_parent_id"] or not row["has_article_no"]:
            status = "metadata_review"

        row["child_status"] = status
        updated.append(row)

    return updated


all_children_final = add_child_quality_flags(all_children)
df_children_final = pd.DataFrame(all_children_final)

print("child_status 분포")
display(df_children_final["child_status"].value_counts().reset_index())

display(df_children_final[[
    "law_name",
    "article_no",
    "article_title",
    "child_index",
    "split_strategy",
    "text_length",
    "child_status",
]].head(30))

child_status 분포


,child_status,count
0,ok,1682
1,short_child_review,472
2,long_child_review,1


,law_name,article_no,article_title,child_index,split_strategy,text_length,child_status
0,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제1조,목적,1,whole_article,81,ok
1,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,정의,1,paragraph_circled,1033,ok
2,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,정의,2,paragraph_circled,1075,ok
3,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,정의,3,paragraph_circled,103,ok
4,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,정의,4,paragraph_circled,130,ok
5,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,정의,5,paragraph_circled,57,short_child_review
6,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,정의,6,paragraph_circled,621,ok
7,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,정의,7,paragraph_circled,91,ok
8,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,정의,8,paragraph_circled,290,ok
9,금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402),제2조,정의,9,paragraph_circled,86,ok


In [17]:
# children.jsonl 저장
save_jsonl(all_children_final, CHILDREN_PATH)

print("children.jsonl 저장 완료:", CHILDREN_PATH)
print("저장 row 수:", len(all_children_final))

children.jsonl 저장 완료: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\children.jsonl
저장 row 수: 2155


In [18]:
# children.jsonl 재로드 검증
reloaded_children = load_jsonl(CHILDREN_PATH)

print("재로드 child row 수:", len(reloaded_children))
pprint(reloaded_children[0] if reloaded_children else None)

assert len(reloaded_children) == len(all_children_final), "저장 row 수와 재로드 row 수가 다릅니다."
assert all("chunk_id" in row for row in reloaded_children), "chunk_id 누락 row가 있습니다."
assert all("parent_id" in row for row in reloaded_children), "parent_id 누락 row가 있습니다."
assert all("text" in row for row in reloaded_children), "text 누락 row가 있습니다."

print("[OK] children.jsonl 재로드 검증 완료")

재로드 child row 수: 2155
{'article_no': '제1조',
 'article_title': '목적',
 'child_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
 'child_index': 1,
 'child_status': 'ok',
 'child_text': '이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한\n'
               '사항을 규정함을 목적으로 한다.',
 'chunk_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
 'doc_code': 'financial_consumer_supervisory_regulation',
 'document_priority': 3,
 'document_type': 'supervisory_regulation',
 'effective_date': '2026.4.2.',
 'has_article_no': True,
 'has_page': True,
 'has_parent_id': True,
 'is_long_child': False,
 'is_short_child': False,
 'item_no': '',
 'keywords': [],
 'law_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)',
 'page': 1,
 'page_end': 1,
 'page_start': 1,
 'paragraph_no': '',
 'parent_id': 'financial_consumer_supervisory_regulation__article_1',
 'parent_text': '제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 '
          

In [19]:
# 검색 seed 기준 태그 확인
SEED_QUERIES = [
    "누구나 승인",
    "최저금리",
    "수수료",
    "원금보장",
    "확정수익",
    "설명의무",
    "부당권유",
]

for query in SEED_QUERIES:
    matched = [
        row for row in all_children_final
        if any(token in row["text"] for token in query.split())
        or query in row["text"]
    ]

    print("=" * 100)
    print("query:", query)
    print("matched children:", len(matched))

    for row in matched[:5]:
        print(
            f"- {row['law_name']} {row['article_no']}({row['article_title']}) "
            f"page={row['page']} strategy={row['split_strategy']} "
            f"tags={row['risk_tags']}"
        )
        print(row["text"][:200].replace("\n", " "))

query: 누구나 승인
matched children: 98
- 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제22조(금융상품판매대리ㆍ중개업자의 금지행위) page=20 strategy=numbered_item tags=['approval_misleading', 'explanation_duty', 'principal_guarantee_misleading']
제22조(금융상품판매대리ㆍ중개업자의 금지행위) 6. 「방송법」 제9조제5항 단서에 따라 상품소개와 판매에 관한 전문편성을 행하는 방송채널사용사업을 승인받은 금융상품판매대리ㆍ중개업자(보장성 상품을 취급하는 자에 한정한다)가 보장성 상품에 관한 금융상품판매대 리ㆍ중개업을 영위할 수 없는 개인으로 하여금 같은 법 제2조제1호에 따른 방송을 통해 그 금융상품을 설
- 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제10조(내부통제기준) page=7 strategy=paragraph_circled tags=['approval_misleading']
제10조(내부통제기준) ③ 금융상품판매업자등은 법 제16조제2항에 따라 내부통제기준을 제정ㆍ변경하는 경우 이사회(이사회가 없는 경우 로서 금융위원회가 정하여 고시하는 경우에는 금융상품판매업자등의 대표자 또는 그 국내지점의 대표자가 참여하 는 내부 의사결정기구로 한다)의 승인을 받아야 한다. 다만, 금융위원회가 정하여 고시하는 경미한 사항을 변경하는 경우에는
- 여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506) 제4조의2(인력ㆍ물적시설의 유지 등) page=4 strategy=whole_article tags=['approval_misleading']
제4조의2(인력ㆍ물적시설의 유지 등) 금융위는 법 제6조의2 단서의 규정에 의한 승인을 함에 있어 승인신청일로부 터 60일내에 시행령 제6조의3제6항 각 호에서 정한 요건을 충족하는지 여부를 확인한 후 승인여부를 결정하고 그 

In [20]:
# 디버그 요약 저장
child_summary = {
    "parent_count": len(parents),
    "child_count": len(all_children_final),
    "build_error_count": len(build_errors),
    "children_path": str(CHILDREN_PATH),
    "length_summary": length_summary,
    "split_strategy_counts": (
        df_children_final["split_strategy"].value_counts().to_dict()
        if not df_children_final.empty
        else {}
    ),
    "child_status_counts": (
        df_children_final["child_status"].value_counts().to_dict()
        if not df_children_final.empty
        else {}
    ),
    "document_type_counts": (
        df_children_final["document_type"].value_counts().to_dict()
        if not df_children_final.empty
        else {}
    ),
    "build_errors": build_errors,
}

summary_path = DEBUG_DIR / "child_chunk_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(child_summary, f, ensure_ascii=False, indent=2)

print("child chunk summary 저장:", summary_path)
pprint(child_summary)

child chunk summary 저장: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\debug_child_chunks\child_chunk_summary.json
{'build_error_count': 0,
 'build_errors': [],
 'child_count': 2155,
 'child_status_counts': {'long_child_review': 1,
                         'ok': 1682,
                         'short_child_review': 472},
 'children_path': 'c:\\Users\\USER\\Desktop\\complypilot-jb\\data\\retrieval\\children.jsonl',
 'document_type_counts': {'enforcement_decree': 230,
                          'law': 1017,
                          'supervisory_regulation': 908},
 'length_summary': {'avg_length': 180.91,
                    'child_count': 2155,
                    'long_gt_3000': 1,
                    'long_gt_5000': 0,
                    'max_length': 3859,
                    'min_length': 11,
                    'short_lt_80': 472},
 'parent_count': 608,
 'split_strategy_counts': {'numbered_item': 350,
                           'paragraph_circled': 1669,
                       

In [21]:
# 최종 체크
print("=" * 100)
print("03_build_parent_child_chunks 최종 체크")
print("=" * 100)

checks = {
    "parents_count_gt_0": len(parents) > 0,
    "children_count_gt_0": len(all_children_final) > 0,
    "children_jsonl_exists": CHILDREN_PATH.exists(),
    "children_reload_match": len(reloaded_children) == len(all_children_final),
    "no_build_errors": len(build_errors) == 0,
    "no_missing_parent_reference": len(missing_parent_ids) == 0,
    "no_duplicate_chunk_id": len(duplicated_chunk_ids) == 0,
    "has_required_metadata": all(
        row.get("law_name")
        and row.get("article_no")
        and row.get("parent_id")
        and row.get("chunk_id")
        for row in all_children_final
    ),
}

pprint(checks)

if all(checks.values()):
    print("[OK] parent-child chunk artifact 생성 완료")
else:
    print("[WARN] 일부 체크가 실패했습니다. 위 결과를 보고 보완이 필요합니다.")

03_build_parent_child_chunks 최종 체크
{'children_count_gt_0': True,
 'children_jsonl_exists': True,
 'children_reload_match': True,
 'has_required_metadata': True,
 'no_build_errors': True,
 'no_duplicate_chunk_id': True,
 'no_missing_parent_reference': True,
 'parents_count_gt_0': True}
[OK] parent-child chunk artifact 생성 완료
